# Music Genre Classification - Supervised Learning Models

**Predictive Question**: Predict music genre from audio features  
**Target Variable**: music_genre (multi-class classification)  
**Dataset**: processed_musicData.csv

**Model Comparison**:
1. Random Forest
2. SVM
3. XGBoost  
4. Neural Network
5. AdaBoost
6. Logistic Regression

## 1. Import Libraries and Load Data

In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import joblib

# XGBoost
try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
except ImportError:
    print("XGBoost not available, will skip this model")
    XGBOOST_AVAILABLE = False

print("All libraries imported successfully")

All libraries imported successfully


In [7]:
# Load data
print("Loading data...")
df = pd.read_csv('processed_musicData.csv')

# Prepare features and target variable
feature_cols = [col for col in df.columns if col not in ['instance_id', 'obtained_date', 'music_genre']]
X = df[feature_cols].fillna(df[feature_cols].median())
y = df['music_genre']

print(f"Data shape: {X.shape}")
print(f"Number of features: {X.shape[1]}")
print(f"Number of samples: {X.shape[0]}")
print(f"Number of classes: {y.nunique()}")

# Show class distribution
print("\nClass distribution:")
print(y.value_counts().sort_index())

Loading data...
Data shape: (50000, 46)
Number of features: 46
Number of samples: 50000
Number of classes: 10

Class distribution:
music_genre
0.0    5000
1.0    5000
2.0    5000
3.0    5000
4.0    5000
5.0    5000
6.0    5000
7.0    5000
8.0    5000
9.0    5000
Name: count, dtype: int64


In [8]:
# Train-test split
print("Creating train-test split...")
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Feature scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")
print("Feature scaling completed")

Creating train-test split...
Training set: 40000 samples
Test set: 10000 samples
Feature scaling completed


## 2. Model 1: Random Forest

In [ ]:
print("="*50)
print("Training Random Forest Model")
print("="*50)

# Define model
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=20,
    min_samples_split=2,
    random_state=42,
    class_weight='balanced',
    n_jobs=-1
)

# Train model
print("Training...")
rf_model.fit(X_train, y_train)

# Predict
rf_pred = rf_model.predict(X_test)

# Evaluate
rf_accuracy = accuracy_score(y_test, rf_pred)
rf_f1_macro = f1_score(y_test, rf_pred, average='macro')
rf_f1_weighted = f1_score(y_test, rf_pred, average='weighted')

print(f"Random Forest Results:")
print(f"  Accuracy: {rf_accuracy:.4f}")
print(f"  F1-macro: {rf_f1_macro:.4f}")
print(f"  F1-weighted: {rf_f1_weighted:.4f}")

# Cross-validation
print("\nPerforming 5-fold cross-validation...")
cv_scores = cross_val_score(rf_model, X_train, y_train, cv=5, scoring='f1_macro', n_jobs=-1)
print(f"Cross-validation F1-macro: {cv_scores.mean():.4f} (+/- {cv_scores.std() * 2:.4f})")

Training Random Forest Model
Training...
Random Forest Results:
  Accuracy: 0.5598
  F1-macro: 0.5584
  F1-weighted: 0.5584

Performing 5-fold cross-validation...
Cross-validation F1-macro: 0.5500 (+/- 0.0113)


## 3. Model 2: Support Vector Machine

In [ ]:
print("="*50)
print("Training SVM Model")
print("="*50)

# Define model - using scaled data
svm_model = SVC(
    C=1.0,
    kernel='rbf',
    gamma='scale',
    random_state=42,
    class_weight='balanced'
)

# Train model
print("Training... (this may take a few minutes)")
svm_model.fit(X_train_scaled, y_train)

# Predict
svm_pred = svm_model.predict(X_test_scaled)

# Evaluate
svm_accuracy = accuracy_score(y_test, svm_pred)
svm_f1_macro = f1_score(y_test, svm_pred, average='macro')
svm_f1_weighted = f1_score(y_test, svm_pred, average='weighted')

print(f"SVM Results:")
print(f"  Accuracy: {svm_accuracy:.4f}")
print(f"  F1-macro: {svm_f1_macro:.4f}")
print(f"  F1-weighted: {svm_f1_weighted:.4f}")

# Cross-validation
print("\nPerforming 5-fold cross-validation...")
cv_scores = cross_val_score(svm_model, X_train_scaled, y_train, cv=5, scoring='f1_macro', n_jobs=-1)
print(f"Cross-validation F1-macro: {cv_scores.mean():.4f} (+/- {cv_scores.std() * 2:.4f})")

Training SVM Model
Training... (this may take a few minutes)
SVM Results:
  Accuracy: 0.5740
  F1-macro: 0.5716
  F1-weighted: 0.5716

Performing 5-fold cross-validation...


## 4. Model 3: XGBoost

In [ ]:
if XGBOOST_AVAILABLE:
    print("="*50)
    print("Training XGBoost Model")
    print("="*50)

    # Define model
    xgb_model = XGBClassifier(
        n_estimators=200,
        learning_rate=0.1,
        max_depth=6,
        random_state=42,
        eval_metric='mlogloss',
        objective='multi:softprob',
        n_jobs=-1,
        verbosity=0
    )

    # Train model
    print("Training...")
    xgb_model.fit(X_train, y_train)

    # Predict
    xgb_pred = xgb_model.predict(X_test)

    # Evaluate
    xgb_accuracy = accuracy_score(y_test, xgb_pred)
    xgb_f1_macro = f1_score(y_test, xgb_pred, average='macro')
    xgb_f1_weighted = f1_score(y_test, xgb_pred, average='weighted')

    print(f"XGBoost Results:")
    print(f"  Accuracy: {xgb_accuracy:.4f}")
    print(f"  F1-macro: {xgb_f1_macro:.4f}")
    print(f"  F1-weighted: {xgb_f1_weighted:.4f}")

    # Cross-validation
    print("\nPerforming 5-fold cross-validation...")
    cv_scores = cross_val_score(xgb_model, X_train, y_train, cv=5, scoring='f1_macro', n_jobs=-1)
    print(f"Cross-validation F1-macro: {cv_scores.mean():.4f} (+/- {cv_scores.std() * 2:.4f})")
else:
    print("XGBoost not available, skipping this model")
    xgb_accuracy = xgb_f1_macro = xgb_f1_weighted = 0.0

## 5. Model 4: Neural Network

In [ ]:
print("="*50)
print("Training Neural Network Model")
print("="*50)

# Define model - using scaled data
nn_model = MLPClassifier(
    hidden_layer_sizes=(100, 50),
    activation='relu',
    alpha=0.001,
    learning_rate='adaptive',
    max_iter=200,
    random_state=42,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=10
)

# Train model
print("Training...")
nn_model.fit(X_train_scaled, y_train)

# Predict
nn_pred = nn_model.predict(X_test_scaled)

# Evaluate
nn_accuracy = accuracy_score(y_test, nn_pred)
nn_f1_macro = f1_score(y_test, nn_pred, average='macro')
nn_f1_weighted = f1_score(y_test, nn_pred, average='weighted')

print(f"Neural Network Results:")
print(f"  Accuracy: {nn_accuracy:.4f}")
print(f"  F1-macro: {nn_f1_macro:.4f}")
print(f"  F1-weighted: {nn_f1_weighted:.4f}")

# Cross-validation
print("\nPerforming 5-fold cross-validation...")
cv_scores = cross_val_score(nn_model, X_train_scaled, y_train, cv=5, scoring='f1_macro', n_jobs=-1)
print(f"Cross-validation F1-macro: {cv_scores.mean():.4f} (+/- {cv_scores.std() * 2:.4f})")

## 6. Model 5: AdaBoost

In [ ]:
print("="*50)
print("Training AdaBoost Model")
print("="*50)

# Define model
ada_model = AdaBoostClassifier(
    n_estimators=200,
    learning_rate=1.0,
    random_state=42,
    algorithm='SAMME'
)

# Train model
print("Training...")
ada_model.fit(X_train, y_train)

# Predict
ada_pred = ada_model.predict(X_test)

# Evaluate
ada_accuracy = accuracy_score(y_test, ada_pred)
ada_f1_macro = f1_score(y_test, ada_pred, average='macro')
ada_f1_weighted = f1_score(y_test, ada_pred, average='weighted')

print(f"AdaBoost Results:")
print(f"  Accuracy: {ada_accuracy:.4f}")
print(f"  F1-macro: {ada_f1_macro:.4f}")
print(f"  F1-weighted: {ada_f1_weighted:.4f}")

# Cross-validation
print("\nPerforming 5-fold cross-validation...")
cv_scores = cross_val_score(ada_model, X_train, y_train, cv=5, scoring='f1_macro', n_jobs=-1)
print(f"Cross-validation F1-macro: {cv_scores.mean():.4f} (+/- {cv_scores.std() * 2:.4f})")

## 7. Model 6: Logistic Regression

In [ ]:
print("="*50)
print("Training Logistic Regression Model")
print("="*50)

# Define model - using scaled data
lr_model = LogisticRegression(
    max_iter=1000,
    random_state=42,
    class_weight='balanced',
    solver='lbfgs',
    n_jobs=-1
)

# Train model
print("Training...")
lr_model.fit(X_train_scaled, y_train)

# Predict
lr_pred = lr_model.predict(X_test_scaled)

# Evaluate
lr_accuracy = accuracy_score(y_test, lr_pred)
lr_f1_macro = f1_score(y_test, lr_pred, average='macro')
lr_f1_weighted = f1_score(y_test, lr_pred, average='weighted')

print(f"Logistic Regression Results:")
print(f"  Accuracy: {lr_accuracy:.4f}")
print(f"  F1-macro: {lr_f1_macro:.4f}")
print(f"  F1-weighted: {lr_f1_weighted:.4f}")

# Cross-validation
print("\nPerforming 5-fold cross-validation...")
cv_scores = cross_val_score(lr_model, X_train_scaled, y_train, cv=5, scoring='f1_macro', n_jobs=-1)
print(f"Cross-validation F1-macro: {cv_scores.mean():.4f} (+/- {cv_scores.std() * 2:.4f})")

## 8. Model Comparison and Results Summary

In [ ]:
print("="*60)
print("Model Performance Comparison")
print("="*60)

# Create results DataFrame
results = {
    'Model': ['Random Forest', 'SVM', 'XGBoost', 'Neural Network', 'AdaBoost', 'Logistic Regression'],
    'Accuracy': [rf_accuracy, svm_accuracy, xgb_accuracy, nn_accuracy, ada_accuracy, lr_accuracy],
    'F1_Macro': [rf_f1_macro, svm_f1_macro, xgb_f1_macro, nn_f1_macro, ada_f1_macro, lr_f1_macro],
    'F1_Weighted': [rf_f1_weighted, svm_f1_weighted, xgb_f1_weighted, nn_f1_weighted, ada_f1_weighted, lr_f1_weighted]
}

results_df = pd.DataFrame(results)
results_df = results_df.sort_values('F1_Macro', ascending=False)

print(results_df.round(4))

# Find best model
best_model = results_df.iloc[0]['Model']
best_f1 = results_df.iloc[0]['F1_Macro']

print(f"\nBest model: {best_model} (F1-Macro: {best_f1:.4f})")

In [ ]:
# Visualize results
plt.figure(figsize=(12, 6))

# Subplot 1: Model performance comparison
plt.subplot(1, 2, 1)
x = np.arange(len(results_df))
width = 0.25

plt.bar(x - width, results_df['Accuracy'], width, label='Accuracy', alpha=0.8)
plt.bar(x, results_df['F1_Macro'], width, label='F1-Macro', alpha=0.8)
plt.bar(x + width, results_df['F1_Weighted'], width, label='F1-Weighted', alpha=0.8)

plt.xlabel('Models')
plt.ylabel('Score')
plt.title('Model Performance Comparison')
plt.xticks(x, results_df['Model'], rotation=45)
plt.legend()
plt.ylim(0, 1)

# Subplot 2: Class distribution
plt.subplot(1, 2, 2)
class_counts = y.value_counts().sort_index()
plt.pie(class_counts.values, labels=class_counts.index, autopct='%1.1f%%')
plt.title('Class Distribution')

plt.tight_layout()
plt.show()

# Save plot
plt.savefig('model_comparison.png', dpi=300, bbox_inches='tight')
print("Plot saved: model_comparison.png")

In [ ]:
# Save results and models
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# 1. Save results CSV
results_file = f'model_results_{timestamp}.csv'
results_df.to_csv(results_file, index=False)
print(f"Results saved: {results_file}")

# 2. Save models
model_files = {
    'random_forest': rf_model,
    'svm': svm_model,
    'neural_network': nn_model,
    'adaboost': ada_model,
    'logistic_regression': lr_model
}

if XGBOOST_AVAILABLE:
    model_files['xgboost'] = xgb_model

for name, model in model_files.items():
    filename = f'{name}_model_{timestamp}.pkl'
    joblib.dump(model, filename)
    print(f"Model saved: {filename}")

# 3. Save feature scaler
scaler_file = f'scaler_{timestamp}.pkl'
joblib.dump(scaler, scaler_file)
print(f"Feature scaler saved: {scaler_file}")

# 4. Save summary report
summary_file = f'summary_report_{timestamp}.txt'
with open(summary_file, 'w', encoding='utf-8') as f:
    f.write("Music Genre Classification - Supervised Learning Models Comparison\n")
    f.write("="*60 + "\n\n")
    f.write(f"Dataset: {len(X)} samples, {len(X.columns)} features\n")
    f.write(f"Number of classes: {y.nunique()}\n")
    f.write(f"Train/Test split: {len(X_train)}/{len(X_test)}\n\n")
    
    f.write("Model Performance Results:\n")
    f.write("-"*30 + "\n")
    for _, row in results_df.iterrows():
        f.write(f"{row['Model']}:\n")
        f.write(f"  Accuracy: {row['Accuracy']:.4f}\n")
        f.write(f"  F1-Macro: {row['F1_Macro']:.4f}\n")
        f.write(f"  F1-Weighted: {row['F1_Weighted']:.4f}\n\n")
    
    f.write(f"Best model: {best_model} (F1-Macro: {best_f1:.4f})\n")

print(f"Summary report saved: {summary_file}")
print("\nAll files saved successfully!")

In [ ]:
print("\n" + "="*60)
print("Final Summary")
print("="*60)
print(f"Dataset: {len(X)} samples, {len(X.columns)} features, {y.nunique()} classes")
print(f"Training set: {len(X_train)} samples")
print(f"Test set: {len(X_test)} samples")
print("\nTrained models:")
print("1. Random Forest - Tree ensemble method")
print("2. SVM - Kernel method")
print("3. XGBoost - Gradient boosting" if True else "3. XGBoost - Not installed")
print("4. Neural Network - Multi-layer perceptron")
print("5. AdaBoost - Adaptive boosting")
print("6. Logistic Regression - Linear model")
print(f"\nBest model: {best_model}")
print(f"Best F1-Macro score: {best_f1:.4f}")
print("\nAll results and models have been saved to files!")